# Spread + Mid Prediction

optimal bps with oracle or BTCUSDT: 0.7055


In [1]:
# ================================
# 1. Clone repo
# ================================
!git clone https://github.com/JosephZacharyGawlik/Ultramarin.git
%cd Ultramarin
!git fetch origin
!git checkout -b spread origin/spread


# ================================
# 2. Install dependencies
# ================================
!pip install --upgrade pip
!pip install .


# ================================
# 3. Mount Google Drive
# ================================
from google.colab import drive
drive.mount('/content/drive')


# ================================
# 4. Set data path
# ================================
DATA_PATH = "/content/drive/MyDrive/Ultramarin Data/data"

import os
print("Files in dataset folder:")
print(os.listdir(DATA_PATH))


# ================================
# 5. GPU check
# ================================
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

fatal: destination path 'Ultramarin' already exists and is not an empty directory.
/content/Ultramarin
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 12 (delta 8), reused 10 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (12/12), 51.99 KiB | 1.49 MiB/s, done.
From https://github.com/JosephZacharyGawlik/Ultramarin
   2eb21e2..94090bb  spread     -> origin/spread
   30cd31e..9371440  potentially-improve-model -> origin/potentially-improve-model
fatal: A branch named 'spread' already exists.
Processing ./.
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
ERROR: Failed to build 'file:///content/Ultramarin'

In [1]:
import numpy as np
from pathlib import Path
import pandas as pd
import torch
import random
from utils.utils import *
from data.simulate_walk_the_book import *
from utils.datastuff import TrainCfg
from utils.train import train_val
from utils.test import generate_test_loader, generate_test_predictions
from data.simulate_walk_the_book import simulate_walk_the_book
import warnings


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# Fix randomness for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

warnings.filterwarnings("ignore", category=UserWarning)

device: cuda


In [2]:
import sys, os

# Paths and volume_to_fill
# root_path = DATA_PATH
root_path = "../data"
root = Path(root_path) if Path(root_path).exists() else Path.cwd()

import sys
if str(root / "src") not in sys.path:
    sys.path.append(str(root / "src"))

data_assets = [folder_name for folder_name in os.listdir(root) if "USDT" in folder_name]
data_asset = data_assets[0]

symbol_dir = root / data_asset
X_path = symbol_dir / "X_train.parquet"
Y_path = symbol_dir / "y_train.parquet"
X_test_path = symbol_dir / "X_test.parquet"
vol_path = symbol_dir / "vol_to_fill.txt"

volume_to_fill = None
if vol_path.exists():
    import re
    with open(vol_path) as f:
        m = re.search(r"([\d.]+)", f.read())
    if m:
        volume_to_fill = float(m.group(1))
print("data asset:", data_asset)
print("volume_to_fill:", volume_to_fill)


data asset: BTCUSDT
volume_to_fill: 4.0


In [3]:
# DeepLOB only take LOB features as input
LOB_COLS = []
for i in range(1, 6):
    LOB_COLS.append(f"ask_price_{i}")
    LOB_COLS.append(f"ask_vol_{i}")
    LOB_COLS.append(f"bid_price_{i}")
    LOB_COLS.append(f"bid_vol_{i}")

FEATURE_COLS = LOB_COLS + ["target"]

# Verification print
print(f"CNN Input Width: 4 (Columns: Price/Vol/Price/Vol)")
print(f"CNN Input Height: 5 (Rows: Levels 1 through 5)")
print(f"Total Features: {len(FEATURE_COLS)} (20 LOB + mid_price)")

CNN Input Width: 4 (Columns: Price/Vol/Price/Vol)
CNN Input Height: 5 (Rows: Levels 1 through 5)
Total Features: 21 (20 LOB + mid_price)


In [4]:
ASK_PRICE_COLS = ['ask_price_1', 'ask_price_2', 'ask_price_3', 'ask_price_4', 'ask_price_5']
ASK_VOL_COLS = ['ask_vol_1', 'ask_vol_2', 'ask_vol_3', 'ask_vol_4', 'ask_vol_5']
BID_PRICE_COLS = ['bid_price_1', 'bid_price_2', 'bid_price_3', 'bid_price_4', 'bid_price_5']
BID_VOL_COLS = ['bid_vol_1', 'bid_vol_2', 'bid_vol_3', 'bid_vol_4', 'bid_vol_5']

# Model Training

In [9]:
# --- Execution ---
config = TrainCfg(

    # Hyperparameters
    epochs = 100, 
    batch_size = 32,
    lr = 1e-3,
    weight_decay = 1e-5,
    smooth_lambda = 0.01,
    
    # Windows
    input_window = 600,   # Look-back
    target_window = 60,   # Horizon
    val_ratio = 0.2,

    # Environment
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    
    # Features & Data
    x_path = X_path,
    y_path = Y_path,
    x_test_path = X_test_path,
    feature_cols = FEATURE_COLS,
)

model, train_scalers, val_loader, val_id_map, processor = train_val(config)

Epoch 01 | Train: 0.717707 | Val: 0.782718
Epoch 02 | Train: 0.268114 | Val: 0.719583
Epoch 03 | Train: 0.097151 | Val: 0.840728
Epoch 04 | Train: 0.049305 | Val: 0.553976
Epoch 05 | Train: 0.029977 | Val: 0.351999
Epoch 06 | Train: 0.028182 | Val: 0.294311
Epoch 07 | Train: 0.021197 | Val: 0.374147
Epoch 08 | Train: 0.018100 | Val: 0.228330
Epoch 09 | Train: 0.014495 | Val: 0.396297
Epoch 10 | Train: 0.014510 | Val: 0.254775
Epoch 11 | Train: 0.013537 | Val: 0.498291
Epoch 12 | Train: 0.011721 | Val: 0.225291
Epoch 13 | Train: 0.011599 | Val: 0.270083
Epoch 14 | Train: 0.011163 | Val: 0.235609
Epoch 15 | Train: 0.011142 | Val: 0.364796
Epoch 16 | Train: 0.010373 | Val: 0.267200
Epoch 17 | Train: 0.009845 | Val: 0.178508
Epoch 18 | Train: 0.008982 | Val: 0.203469
Epoch 19 | Train: 0.008799 | Val: 0.218079
Epoch 20 | Train: 0.008946 | Val: 0.234520
Epoch 21 | Train: 0.008695 | Val: 0.205917
Epoch 22 | Train: 0.009138 | Val: 0.267317
Epoch 23 | Train: 0.008447 | Val: 0.326701
Epoch 24 | 

# Mid Shift Model and BPS

In [10]:
config.target = "mid shift"
config.epochs = 200

model_shift, train_scalers_shift, val_loader_shift, val_id_map_shift, processor_shift = train_val(config)

Epoch 01 | Train: 0.678174 | Val: 1.485002
Epoch 02 | Train: 0.317325 | Val: 0.749770
Epoch 03 | Train: 0.078086 | Val: 0.465001
Epoch 04 | Train: 0.055694 | Val: 0.407281
Epoch 05 | Train: 0.033847 | Val: 0.312975
Epoch 06 | Train: 0.022151 | Val: 0.493046
Epoch 07 | Train: 0.018696 | Val: 0.339085
Epoch 08 | Train: 0.015609 | Val: 0.274392
Epoch 09 | Train: 0.015659 | Val: 0.234531
Epoch 10 | Train: 0.013550 | Val: 0.272035
Epoch 11 | Train: 0.013270 | Val: 0.274481
Epoch 12 | Train: 0.012258 | Val: 0.182062
Epoch 13 | Train: 0.011647 | Val: 0.486115
Epoch 14 | Train: 0.012737 | Val: 0.882706
Epoch 15 | Train: 0.010913 | Val: 0.254998
Epoch 16 | Train: 0.010148 | Val: 0.214314
Epoch 17 | Train: 0.009286 | Val: 0.196297
Epoch 18 | Train: 0.008980 | Val: 0.180166
Epoch 19 | Train: 0.008777 | Val: 0.298717
Epoch 20 | Train: 0.008246 | Val: 0.288032
Epoch 21 | Train: 0.007993 | Val: 0.157171
Epoch 22 | Train: 0.008231 | Val: 0.364793
Epoch 23 | Train: 0.008177 | Val: 0.339313
Epoch 24 | 

## Cell

In [11]:
# Implementation error for the model
 
# --- 1. Generate predictions using val_loader ---
model_shift.eval()
all_preds = []

last_mid_train = train_scalers_shift["last_mid_train"]
last_mid_val = train_scalers_shift["last_mid_val"]

mid_shift_idx = {col: i for i, col in enumerate(FEATURE_COLS)}["target"]

val_mid_shift_mean = train_scalers_shift["val_feat_means"][:, :, mid_shift_idx]
val_mid_shift_std  = train_scalers_shift["val_feat_stds"][:, :, mid_shift_idx]

with torch.no_grad():
    for x_batch, y_batch, target in val_loader_shift:
        x_batch = x_batch.to(config.device)
        pred = model_shift(x_batch, y_teacher=None)
        all_preds.append(pred.cpu().numpy())

val_preds = np.concatenate(all_preds, axis=0)  # [num_val_ids, 60]

# denormalize mid shifts
mid_shift_m = val_mid_shift_mean.squeeze(0).cpu().numpy()[:, None]
mid_shift_s = val_mid_shift_std.squeeze(0).cpu().numpy()[:, None]

val_preds = val_preds * mid_shift_s + mid_shift_m

# convert mid shift -> predicted mid
val_preds = val_preds + last_mid_val.values[:, None]

# Get validation IDs
val_ids = [int(uid) for idx, uid in val_id_map_shift.cpu().numpy()]
print(f"Validation predictions shape: {val_preds.shape}")
print(f"Number of validation instruments: {len(val_ids)}")

# --- 2. Reload raw Y validation data to simulate walking the book ---
Y_raw = pd.read_parquet(Y_path).sort_values(["anonymized_id", "time_in_hour"])
Y_val_raw = Y_raw[Y_raw["anonymized_id"].isin(val_ids)].copy()

# --- 3. Column definitions ---
ASK_PRICE_COLS = ['ask_price_1', 'ask_price_2', 'ask_price_3', 'ask_price_4', 'ask_price_5']
ASK_VOL_COLS = ['ask_vol_1', 'ask_vol_2', 'ask_vol_3', 'ask_vol_4', 'ask_vol_5']
BID_PRICE_COLS = ['bid_price_1', 'bid_price_2', 'bid_price_3', 'bid_price_4', 'bid_price_5']
BID_VOL_COLS = ['bid_vol_1', 'bid_vol_2', 'bid_vol_3', 'bid_vol_4', 'bid_vol_5']

# --- 4. Calculate MODEL implementation error ---
model_shift_bps = []

for i, anon_id in enumerate(val_ids):
    df_inst = Y_val_raw[Y_val_raw["anonymized_id"] == anon_id].sort_values("time_in_hour")
    
    if len(df_inst) != 60:
        continue
    
    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
    close_price = df_inst['close'].dropna().iloc[-1]
    
    # Model-based positions
    mid_preds = val_preds[i]
    pred_close = mid_preds[-1]
    
    price_distance = abs(pred_close - mid_preds)
    weights = 1 - price_distance / price_distance.sum()

    weights /= weights.sum()
    
    model_positions = weights * volume_to_fill
    
    # Simulate
    model_vol, model_avg_price = simulate_walk_the_book(
        model_positions, ask_prices, ask_vols, bid_prices, bid_vols
    )
    
    if model_vol > 0 and not np.isnan(model_avg_price):
        impl_error = np.abs(model_avg_price - close_price) / close_price * 10000
        vol_penalty = min(100.0, volume_to_fill / model_vol)
        model_shift_bps.append(impl_error * vol_penalty)

model_shift_bps = np.array(model_shift_bps)

print(f"\n{'='*50}")
print(f"MODEL IMPLEMENTATION ERROR mid shift")
print(f"{'='*50}")
print(f"Instruments evaluated: {len(model_shift_bps)}")
print(f"Mean:   {model_shift_bps.mean():.4f} bps")
print(f"Median: {np.median(model_shift_bps):.4f} bps")
print(f"Std:    {model_shift_bps.std():.4f} bps")
print(f"Min:    {model_shift_bps.min():.4f} bps")
print(f"Max:    {model_shift_bps.max():.4f} bps")
print(f"{'='*50}")

Validation predictions shape: (141, 60)
Number of validation instruments: 141

MODEL IMPLEMENTATION ERROR mid shift
Instruments evaluated: 141
Mean:   1.7115 bps
Median: 1.2639 bps
Std:    1.6370 bps
Min:    0.0512 bps
Max:    10.4892 bps


# Model-Based Predictions (Spread)

In [12]:
times = pd.read_parquet(Y_path)["time_in_hour"].sort_values(ascending=True).unique()

In [13]:
# train a model for spread
config.target = "spread" 
config.epoch = 200
config.loss = "huber"
# TODO: try using some kind of classification thing for spread

model_s, train_scalers_s, val_loader_s, val_id_map_s, processor_s = train_val(config)

Epoch 01 | Train: 0.447956 | Val: 0.487693
Epoch 02 | Train: 0.411912 | Val: 0.517707
Epoch 03 | Train: 0.405197 | Val: 0.464574
Epoch 04 | Train: 0.402094 | Val: 0.462340


Epoch 05 | Train: 0.399029 | Val: 0.466734
Epoch 06 | Train: 0.397220 | Val: 0.465167
Epoch 07 | Train: 0.396620 | Val: 0.467540
Epoch 08 | Train: 0.395029 | Val: 0.473799
Epoch 09 | Train: 0.394519 | Val: 0.467363
Epoch 10 | Train: 0.392851 | Val: 0.471999
Epoch 11 | Train: 0.391102 | Val: 0.468830
Epoch 12 | Train: 0.389732 | Val: 0.466586
Epoch 13 | Train: 0.389598 | Val: 0.476695
Epoch 14 | Train: 0.387935 | Val: 0.479442
Epoch 15 | Train: 0.387298 | Val: 0.485570
Epoch 16 | Train: 0.387553 | Val: 0.465650
Epoch 17 | Train: 0.384912 | Val: 0.467282
Epoch 18 | Train: 0.382982 | Val: 0.468046
Epoch 19 | Train: 0.381455 | Val: 0.473118
Epoch 20 | Train: 0.381716 | Val: 0.485489
Epoch 21 | Train: 0.381018 | Val: 0.495997
Epoch 22 | Train: 0.380797 | Val: 0.475226
Epoch 23 | Train: 0.377755 | Val: 0.474436
Epoch 24 | Train: 0.377098 | Val: 0.483105
Epoch 25 | Train: 0.377069 | Val: 0.468845
Epoch 26 | Train: 0.375299 | Val: 0.502659
Epoch 27 | Train: 0.373877 | Val: 0.471285
Epoch 28 | 

## Cell

In [14]:
# Implementation error for the spread model
 
# --- 1. Generate predictions using val_loader ---
model_s.eval()
all_preds = []

with torch.no_grad():
    for x_batch, y_batch, target in val_loader:
        x_batch = x_batch.to(config.device)
        pred = model_s(x_batch, y_teacher=None)
        all_preds.append(pred.cpu().numpy())

val_preds = np.concatenate(all_preds, axis=0)  # Shape: [num_val_ids, 60]

# Get validation IDs
val_ids = [int(uid) for idx, uid in val_id_map.cpu().numpy()]
print(f"Validation predictions shape: {val_preds.shape}")
print(f"Number of validation instruments: {len(val_ids)}")

# --- 2. Reload raw Y validation data to simulate walking the book ---
Y_raw = pd.read_parquet(Y_path).sort_values(["anonymized_id", "time_in_hour"])
Y_val_raw = Y_raw[Y_raw["anonymized_id"].isin(val_ids)].copy()

# --- 3. Column definitions ---
ASK_PRICE_COLS = ['ask_price_1', 'ask_price_2', 'ask_price_3', 'ask_price_4', 'ask_price_5']
ASK_VOL_COLS = ['ask_vol_1', 'ask_vol_2', 'ask_vol_3', 'ask_vol_4', 'ask_vol_5']
BID_PRICE_COLS = ['bid_price_1', 'bid_price_2', 'bid_price_3', 'bid_price_4', 'bid_price_5']
BID_VOL_COLS = ['bid_vol_1', 'bid_vol_2', 'bid_vol_3', 'bid_vol_4', 'bid_vol_5']

# --- 4. Calculate MODEL implementation error ---
model_s_bps = []

for i, anon_id in enumerate(val_ids):
    df_inst = Y_val_raw[Y_val_raw["anonymized_id"] == anon_id].sort_values("time_in_hour")
    
    if len(df_inst) != 60:
        continue
    
    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
    close_price = df_inst['close'].dropna().iloc[-1]
    
    # Model-based positions
    spread_preds = val_preds[i]

    # spread weights

    raw_w = 1 / spread_preds

    w_min = raw_w.min()
    w_max = raw_w.max()

    norm_w = (raw_w - w_min) / (w_max - w_min)

    weights = norm_w / norm_w.sum()
    
    model_positions = weights * volume_to_fill
    
    # Simulate
    model_vol, model_avg_price = simulate_walk_the_book(
        model_positions, ask_prices, ask_vols, bid_prices, bid_vols
    )
    
    if model_vol > 0 and not np.isnan(model_avg_price):
        impl_error = np.abs(model_avg_price - close_price) / close_price * 10000
        vol_penalty = min(100.0, volume_to_fill / model_vol)
        model_s_bps.append(impl_error * vol_penalty)

model_s_bps = np.array(model_s_bps)

print(f"\n{'='*50}")
print(f"MODEL IMPLEMENTATION ERROR spread")
print(f"{'='*50}")
print(f"Instruments evaluated: {len(model_s_bps)}")
print(f"Mean:   {model_s_bps.mean():.4f} bps")
print(f"Median: {np.median(model_s_bps):.4f} bps")
print(f"Std:    {model_s_bps.std():.4f} bps")
print(f"Min:    {model_s_bps.min():.4f} bps")
print(f"Max:    {model_s_bps.max():.4f} bps")
print(f"{'='*50}")

Validation predictions shape: (141, 60)
Number of validation instruments: 141

MODEL IMPLEMENTATION ERROR spread
Instruments evaluated: 141
Mean:   1.7395 bps
Median: 1.3346 bps
Std:    1.6883 bps
Min:    0.0001 bps
Max:    11.4110 bps


## Positions

In [11]:
# --- 1. Generate predictions using test_loader ---
model.eval()
test_preds = []

test_loader, scalers, test_map, processor = generate_test_loader(config, processor)

with torch.no_grad():
    for x_batch in test_loader:
        x_batch = x_batch.to(config.device)
        pred = model(x_batch, y_teacher=None)
        test_preds.append(pred.cpu().numpy())

test_preds = np.concatenate(test_preds, axis=0)  # Shape: [num_test_ids, 60]

# Get test IDs
test_ids = [int(uid) for idx, uid in test_map.cpu().numpy()]
print(f"Test predictions shape: {test_preds.shape}")
print(f"Number of testing instruments: {len(test_ids)}")

test_positions = pd.DataFrame()

for i, anon_id in enumerate(test_ids):
    
    # Model-based positions
    spread_preds = test_preds[i]

    # spread weights
    raw_w = 1 / spread_preds

    w_min = raw_w.min()
    w_max = raw_w.max()

    norm_w = (raw_w - w_min) / (w_max - w_min)

    weights = norm_w / norm_w.sum()
    
    model_positions = weights * volume_to_fill

    df = pd.DataFrame({
        'anonymized_id': anon_id,
        'time_in_hour': times,
        'position': model_positions
    })

    test_positions = pd.concat([test_positions, df], ignore_index=True)

test_positions

Test predictions shape: (303, 60)
Number of testing instruments: 303


,anonymized_id,time_in_hour,position
0,5.692284e+16,0 days 00:59:00,0.000000
1,5.692284e+16,0 days 00:59:01,0.004816
2,5.692284e+16,0 days 00:59:02,0.006147
3,5.692284e+16,0 days 00:59:03,0.008299
4,5.692284e+16,0 days 00:59:04,0.010694
...,...,...,...
18175,1.841373e+19,0 days 00:59:55,0.005958
18176,1.841373e+19,0 days 00:59:56,0.004416
18177,1.841373e+19,0 days 00:59:57,0.002910
18178,1.841373e+19,0 days 00:59:58,0.001438


# Model-Based Predictions (Mid)

## Cell

In [15]:
# Implementation error for the model
 
# --- 1. Generate predictions using val_loader ---
model.eval()
all_preds = []

with torch.no_grad():
    for x_batch, y_batch, target in val_loader:
        x_batch = x_batch.to(config.device)
        pred = model(x_batch, y_teacher=None)
        all_preds.append(pred.cpu().numpy())

val_preds = np.concatenate(all_preds, axis=0)  # Shape: [num_val_ids, 60]

# Get validation IDs
val_ids = [int(uid) for idx, uid in val_id_map.cpu().numpy()]
print(f"Validation predictions shape: {val_preds.shape}")
print(f"Number of validation instruments: {len(val_ids)}")

# --- 2. Reload raw Y validation data to simulate walking the book ---
Y_raw = pd.read_parquet(Y_path).sort_values(["anonymized_id", "time_in_hour"])
Y_val_raw = Y_raw[Y_raw["anonymized_id"].isin(val_ids)].copy()

# --- 3. Column definitions ---
ASK_PRICE_COLS = ['ask_price_1', 'ask_price_2', 'ask_price_3', 'ask_price_4', 'ask_price_5']
ASK_VOL_COLS = ['ask_vol_1', 'ask_vol_2', 'ask_vol_3', 'ask_vol_4', 'ask_vol_5']
BID_PRICE_COLS = ['bid_price_1', 'bid_price_2', 'bid_price_3', 'bid_price_4', 'bid_price_5']
BID_VOL_COLS = ['bid_vol_1', 'bid_vol_2', 'bid_vol_3', 'bid_vol_4', 'bid_vol_5']

# --- 4. Calculate MODEL implementation error ---
model_bps = []

for i, anon_id in enumerate(val_ids):
    df_inst = Y_val_raw[Y_val_raw["anonymized_id"] == anon_id].sort_values("time_in_hour")
    
    if len(df_inst) != 60:
        continue
    
    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
    close_price = df_inst['close'].dropna().iloc[-1]
    
    # Model-based positions
    mid_preds = val_preds[i]
    pred_close = mid_preds[-1]
    
    price_distance = abs(pred_close - mid_preds)
    weights = 1 - price_distance / price_distance.sum()

    weights /= weights.sum()
    
    model_positions = weights * volume_to_fill
    
    # Simulate
    model_vol, model_avg_price = simulate_walk_the_book(
        model_positions, ask_prices, ask_vols, bid_prices, bid_vols
    )
    
    if model_vol > 0 and not np.isnan(model_avg_price):
        impl_error = np.abs(model_avg_price - close_price) / close_price * 10000
        vol_penalty = min(100.0, volume_to_fill / model_vol)
        model_bps.append(impl_error * vol_penalty)

model_bps = np.array(model_bps)

print(f"\n{'='*50}")
print(f"MODEL IMPLEMENTATION ERROR mid")
print(f"{'='*50}")
print(f"Instruments evaluated: {len(model_bps)}")
print(f"Mean:   {model_bps.mean():.4f} bps")
print(f"Median: {np.median(model_bps):.4f} bps")
print(f"Std:    {model_bps.std():.4f} bps")
print(f"Min:    {model_bps.min():.4f} bps")
print(f"Max:    {model_bps.max():.4f} bps")
print(f"{'='*50}")

Validation predictions shape: (141, 60)
Number of validation instruments: 141

MODEL IMPLEMENTATION ERROR mid
Instruments evaluated: 141
Mean:   1.7120 bps
Median: 1.2635 bps
Std:    1.6356 bps
Min:    0.0490 bps
Max:    10.4840 bps


## Positions

In [13]:
# --- 1. Generate predictions using test_loader ---
model.eval()
test_preds = []

test_loader, scalers, test_map, processor = generate_test_loader(config, processor)

with torch.no_grad():
    for x_batch in test_loader:
        x_batch = x_batch.to(config.device)
        pred = model(x_batch, y_teacher=None)
        test_preds.append(pred.cpu().numpy())

test_preds = np.concatenate(test_preds, axis=0)  # Shape: [num_test_ids, 60]

# Get test IDs
test_ids = [int(uid) for idx, uid in test_map.cpu().numpy()]
print(f"Test predictions shape: {test_preds.shape}")
print(f"Number of testing instruments: {len(test_ids)}")

test_positions = pd.DataFrame()

for i, anon_id in enumerate(test_ids):
    
    # Model-based positions
    mid_preds = test_preds[i]
    pred_close = mid_preds[-1]
    
    price_distance = abs(pred_close - mid_preds)
    weights = 1 - (price_distance / price_distance.sum())

    weights /= weights.sum()
    
    model_positions = weights * volume_to_fill

    df = pd.DataFrame({
        'anonymized_id': anon_id,
        'time_in_hour': times,
        'position': model_positions
    })

    test_positions = pd.concat([test_positions, df], ignore_index=True)

test_positions

Test predictions shape: (303, 60)
Number of testing instruments: 303


,anonymized_id,time_in_hour,position
0,5.692284e+16,0 days 00:59:00,0.065446
1,5.692284e+16,0 days 00:59:01,0.065510
2,5.692284e+16,0 days 00:59:02,0.065528
3,5.692284e+16,0 days 00:59:03,0.065557
4,5.692284e+16,0 days 00:59:04,0.065590
...,...,...,...
18175,1.841373e+19,0 days 00:59:55,0.067677
18176,1.841373e+19,0 days 00:59:56,0.067708
18177,1.841373e+19,0 days 00:59:57,0.067738
18178,1.841373e+19,0 days 00:59:58,0.067768


# Model-Based Predictions (Mid + Spread)

The ideal model that combines Mid and Spread as evaluated previously was 1/6 mid + 5/6 spread.

In [16]:
# Implementation error for the model
 
# --- 1. Generate predictions using val_loader ---
model.eval()
all_preds = []

with torch.no_grad():
    for x_batch, y_batch, target in val_loader:
        x_batch = x_batch.to(config.device)
        pred = model(x_batch, y_teacher=None)
        all_preds.append(pred.cpu().numpy())

val_preds = np.concatenate(all_preds, axis=0)  # Shape: [num_val_ids, 60]

# --- 1. Generate predictions using val_loader ---
model_s.eval()
all_preds_s = []

with torch.no_grad():
    for x_batch, y_batch, target in val_loader:
        x_batch = x_batch.to(config.device)
        pred = model_s(x_batch, y_teacher=None)
        all_preds_s.append(pred.cpu().numpy())

val_preds_s = np.concatenate(all_preds, axis=0)  # Shape: [num_val_ids, 60]

# Get validation IDs
val_ids = [int(uid) for idx, uid in val_id_map.cpu().numpy()]
print(f"Validation predictions shape: {val_preds.shape}")
print(f"Number of validation instruments: {len(val_ids)}")

# --- 2. Reload raw Y validation data to simulate walking the book ---
Y_raw = pd.read_parquet(Y_path).sort_values(["anonymized_id", "time_in_hour"])
Y_val_raw = Y_raw[Y_raw["anonymized_id"].isin(val_ids)].copy()

# --- 3. Column definitions ---
ASK_PRICE_COLS = ['ask_price_1', 'ask_price_2', 'ask_price_3', 'ask_price_4', 'ask_price_5']
ASK_VOL_COLS = ['ask_vol_1', 'ask_vol_2', 'ask_vol_3', 'ask_vol_4', 'ask_vol_5']
BID_PRICE_COLS = ['bid_price_1', 'bid_price_2', 'bid_price_3', 'bid_price_4', 'bid_price_5']
BID_VOL_COLS = ['bid_vol_1', 'bid_vol_2', 'bid_vol_3', 'bid_vol_4', 'bid_vol_5']

# --- 4. Calculate MODEL implementation error ---
model_ms_bps = []

for i, anon_id in enumerate(val_ids):
    df_inst = Y_val_raw[Y_val_raw["anonymized_id"] == anon_id].sort_values("time_in_hour")
    
    if len(df_inst) != 60:
        continue
    
    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
    close_price = df_inst['close'].dropna().iloc[-1]
    
    # Model-based positions
    mid_preds = val_preds[i]
    pred_close = mid_preds[-1]
    
    price_distance = abs(pred_close - mid_preds)
    weights = 1 - price_distance / price_distance.sum()

    weights /= weights.sum()

    ### spread
    # Model-based positions
    spread_preds = val_preds_s[i]

    # spread weights

    raw_w = 1 / spread_preds

    w_min = raw_w.min()
    w_max = raw_w.max()

    norm_w = (raw_w - w_min) / (w_max - w_min)

    weights_s = norm_w / norm_w.sum()
    ###

    weights = 1/6 * weights + 5/6 * weights_s
    
    model_positions = weights * volume_to_fill
    
    # Simulate
    model_vol, model_avg_price = simulate_walk_the_book(
        model_positions, ask_prices, ask_vols, bid_prices, bid_vols
    )
    
    if model_vol > 0 and not np.isnan(model_avg_price):
        impl_error = np.abs(model_avg_price - close_price) / close_price * 10000
        vol_penalty = min(100.0, volume_to_fill / model_vol)
        model_ms_bps.append(impl_error * vol_penalty)

model_ms_bps = np.array(model_ms_bps)

print(f"\n{'='*50}")
print(f"MODEL IMPLEMENTATION ERROR mid + spread")
print(f"{'='*50}")
print(f"Instruments evaluated: {len(model_ms_bps)}")
print(f"Mean:   {model_ms_bps.mean():.4f} bps")
print(f"Median: {np.median(model_ms_bps):.4f} bps")
print(f"Std:    {model_ms_bps.std():.4f} bps")
print(f"Min:    {model_ms_bps.min():.4f} bps")
print(f"Max:    {model_ms_bps.max():.4f} bps")
print(f"{'='*50}")

Validation predictions shape: (141, 60)
Number of validation instruments: 141

MODEL IMPLEMENTATION ERROR mid + spread
Instruments evaluated: 141
Mean:   1.6219 bps
Median: 1.2514 bps
Std:    1.5031 bps
Min:    0.0002 bps
Max:    8.5305 bps


# TWAP

## Cell

In [17]:
# Implementation error for the baseline (TWAP)

K_SECONDS = 60  # TWAP window: last K seconds

baseline_bps = []

for i, anon_id in enumerate(val_ids):
    df_inst = Y_val_raw[Y_val_raw["anonymized_id"] == anon_id].sort_values("time_in_hour")
    
    if len(df_inst) != 60:
        continue
    
    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
    close_price = df_inst['close'].dropna().iloc[-1]
    
    # Baseline TWAP positions
    baseline_positions = np.zeros(60)
    baseline_positions[-K_SECONDS:] = volume_to_fill / K_SECONDS
    
    # Simulate
    baseline_vol, baseline_avg_price = simulate_walk_the_book(
        baseline_positions, ask_prices, ask_vols, bid_prices, bid_vols
    )
    
    if baseline_vol > 0 and not np.isnan(baseline_avg_price):
        impl_error = np.abs(baseline_avg_price - close_price) / close_price * 10000
        vol_penalty = min(100.0, volume_to_fill / baseline_vol)
        baseline_bps.append(impl_error * vol_penalty)

baseline_bps = np.array(baseline_bps)

print(f"\n{'='*50}")
print(f"BASELINE (TWAP-{K_SECONDS}s) IMPLEMENTATION ERROR")
print(f"{'='*50}")
print(f"Instruments evaluated: {len(baseline_bps)}")
print(f"Mean:   {baseline_bps.mean():.4f} bps")
print(f"Median: {np.median(baseline_bps):.4f} bps")
print(f"Std:    {baseline_bps.std():.4f} bps")
print(f"Min:    {baseline_bps.min():.4f} bps")
print(f"Max:    {baseline_bps.max():.4f} bps")
print(f"{'='*50}")


BASELINE (TWAP-60s) IMPLEMENTATION ERROR
Instruments evaluated: 141
Mean:   1.7176 bps
Median: 1.2783 bps
Std:    1.6430 bps
Min:    0.0400 bps
Max:    10.5569 bps


In [18]:
# Implementation error for the baseline (TWAP)

K_SECONDS = 14  # TWAP window: last K seconds

baseline_14_bps = []

for i, anon_id in enumerate(val_ids):
    df_inst = Y_val_raw[Y_val_raw["anonymized_id"] == anon_id].sort_values("time_in_hour")
    
    if len(df_inst) != 60:
        continue
    
    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
    close_price = df_inst['close'].dropna().iloc[-1]
    
    # Baseline TWAP positions
    baseline_positions = np.zeros(60)
    baseline_positions[-K_SECONDS:] = volume_to_fill / K_SECONDS
    
    # Simulate
    baseline_vol, baseline_avg_price = simulate_walk_the_book(
        baseline_positions, ask_prices, ask_vols, bid_prices, bid_vols
    )
    
    if baseline_vol > 0 and not np.isnan(baseline_avg_price):
        impl_error = np.abs(baseline_avg_price - close_price) / close_price * 10000
        vol_penalty = min(100.0, volume_to_fill / baseline_vol)
        baseline_14_bps.append(impl_error * vol_penalty)

baseline_14_bps = np.array(baseline_14_bps)

print(f"\n{'='*50}")
print(f"BASELINE (TWAP-{K_SECONDS}s) IMPLEMENTATION ERROR")
print(f"{'='*50}")
print(f"Instruments evaluated: {len(baseline_14_bps)}")
print(f"Mean:   {baseline_14_bps.mean():.4f} bps")
print(f"Median: {np.median(baseline_14_bps):.4f} bps")
print(f"Std:    {baseline_14_bps.std():.4f} bps")
print(f"Min:    {baseline_14_bps.min():.4f} bps")
print(f"Max:    {baseline_14_bps.max():.4f} bps")
print(f"{'='*50}")


BASELINE (TWAP-14s) IMPLEMENTATION ERROR
Instruments evaluated: 141
Mean:   1.3232 bps
Median: 1.1836 bps
Std:    1.0940 bps
Min:    0.0000 bps
Max:    9.0855 bps


## Positions

In [15]:
test_positions_twap = pd.DataFrame()

for i, anon_id in enumerate(test_ids):
    
    weights = 1 / 60
    
    model_positions = weights * volume_to_fill

    df = pd.DataFrame({
        'anonymized_id': anon_id,
        'time_in_hour': times,
        'position': model_positions
    })

    test_positions_twap = pd.concat([test_positions_twap, df], ignore_index=True)

test_positions_twap

,anonymized_id,time_in_hour,position
0,5.692284e+16,0 days 00:59:00,0.066667
1,5.692284e+16,0 days 00:59:01,0.066667
2,5.692284e+16,0 days 00:59:02,0.066667
3,5.692284e+16,0 days 00:59:03,0.066667
4,5.692284e+16,0 days 00:59:04,0.066667
...,...,...,...
18175,1.841373e+19,0 days 00:59:55,0.066667
18176,1.841373e+19,0 days 00:59:56,0.066667
18177,1.841373e+19,0 days 00:59:57,0.066667
18178,1.841373e+19,0 days 00:59:58,0.066667


# Predictive_Scheduler Predictions

## Cell

In [19]:
# Implementation error for the predictive scheduler

from predictive_scheduler import build_schedule_from_forecasts, ScheduleConfig

# Configure the scheduler
sched_cfg = ScheduleConfig(
    window=14,       # Only trade in last 14 seconds
    alpha=0.1,       # 10% model, 90% TWAP blend
    price_cap=3.0,   # Cap extreme scores
)

scheduler_bps = []

for i, anon_id in enumerate(val_ids):
    df_inst = Y_val_raw[Y_val_raw["anonymized_id"] == anon_id].sort_values("time_in_hour")
    
    if len(df_inst) != 60:
        continue
    
    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
    close_price = df_inst['close'].dropna().iloc[-1]
    
    # Use predictive_scheduler to build positions
    mid_preds = val_preds[i]
    scheduler_positions = build_schedule_from_forecasts(
        mid_pred=mid_preds,
        volume_to_fill=volume_to_fill,
        spread_pred=None,  # We don't predict spread
        liq_pred=None,     # We don't predict liquidity
        cfg=sched_cfg
    )
    
    # Simulate
    sched_vol, sched_avg_price = simulate_walk_the_book(
        scheduler_positions, ask_prices, ask_vols, bid_prices, bid_vols
    )
    
    if sched_vol > 0 and not np.isnan(sched_avg_price):
        impl_error = np.abs(sched_avg_price - close_price) / close_price * 10000
        vol_penalty = min(100.0, volume_to_fill / sched_vol)
        scheduler_bps.append(impl_error * vol_penalty)

scheduler_bps = np.array(scheduler_bps)

print(f"\n{'='*50}")
print(f"PREDICTIVE SCHEDULER IMPLEMENTATION ERROR")
print(f"{'='*50}")
print(f"Config: window={sched_cfg.window}, alpha={sched_cfg.alpha}")
print(f"Instruments evaluated: {len(scheduler_bps)}")
print(f"Mean:   {scheduler_bps.mean():.4f} bps")
print(f"Median: {np.median(scheduler_bps):.4f} bps")
print(f"Std:    {scheduler_bps.std():.4f} bps")
print(f"Min:    {scheduler_bps.min():.4f} bps")
print(f"Max:    {scheduler_bps.max():.4f} bps")
print(f"{'='*50}")


PREDICTIVE SCHEDULER IMPLEMENTATION ERROR
Config: window=14, alpha=0.1
Instruments evaluated: 141
Mean:   1.3174 bps
Median: 1.2091 bps
Std:    1.1127 bps
Min:    0.0003 bps
Max:    10.0003 bps


## Positions

In [ ]:
# Implementation error for the predictive scheduler

from predictive_scheduler import build_schedule_from_forecasts, ScheduleConfig

# Configure the scheduler
sched_cfg = ScheduleConfig(
    window=14,       # Only trade in last 14 seconds
    alpha=0.1,       # 10% model, 90% TWAP blend
    price_cap=3.0,   # Cap extreme scores
)

test_positions_scheduler = pd.DataFrame()

for i, anon_id in enumerate(test_ids):
    
    # Use predictive_scheduler to build positions
    mid_preds = test_preds[i]
    
    scheduler_positions = build_schedule_from_forecasts(
        mid_pred=mid_preds,
        volume_to_fill=volume_to_fill,
        spread_pred=None,  # We don't predict spread
        liq_pred=None,     # We don't predict liquidity
        cfg=sched_cfg
    )

    df = pd.DataFrame({
        'anonymized_id': anon_id,
        'time_in_hour': times,
        'position': scheduler_positions
    })

    test_positions_scheduler = pd.concat([test_positions_scheduler, df], ignore_index=True)

test_positions_scheduler

,anonymized_id,time_in_hour,position
0,5.692284e+16,0 days 00:59:00,0.000000
1,5.692284e+16,0 days 00:59:01,0.000000
2,5.692284e+16,0 days 00:59:02,0.000000
3,5.692284e+16,0 days 00:59:03,0.000000
4,5.692284e+16,0 days 00:59:04,0.000000
...,...,...,...
18175,1.841373e+19,0 days 00:59:55,0.286168
18176,1.841373e+19,0 days 00:59:56,0.286168
18177,1.841373e+19,0 days 00:59:57,0.286168
18178,1.841373e+19,0 days 00:59:58,0.286168


# Save Positions

In [ ]:
len(test_positions["anonymized_id"].unique())

303

In [ ]:
base_path = Path(f"positions_joe/{data_asset}")
base_path.mkdir(parents=True, exist_ok=True)

# 2. Map the filenames to the actual DataFrame objects
save_map = {
    "midprice.parquet": test_positions,
    "twap.parquet": test_positions_twap,
    "scheduler.parquet": test_positions_scheduler
}

# 3. Save each with a quick confirmation
for filename, df in save_map.items():
    if df is not None and not df.empty:
        target_path = base_path / filename
        df.to_parquet(target_path, index=False, engine='pyarrow')
        print(f"✅ Saved {len(df)} rows to: {target_path}")
    else:
        print(f"⚠️ Skipping {filename}: DataFrame is empty or None.")

✅ Saved 18180 rows to: positions_joe/BTCUSDT/midprice.parquet
✅ Saved 18180 rows to: positions_joe/BTCUSDT/twap.parquet
✅ Saved 18180 rows to: positions_joe/BTCUSDT/scheduler.parquet


# Evaluation Metrics

We want a file of this format:

MSE, R2, BPS_MID, BPS_TWAP, BPS_SCHEDULER 

In [21]:
model.eval()

# Per-instrument normalization stats for VALIDATION set
mid_idx = {col: i for i, col in enumerate(FEATURE_COLS)}["target"]
val_mid_mean = train_scalers["val_feat_means"][:, :, mid_idx]  # [1, num_val_ids]
val_mid_std  = train_scalers["val_feat_stds"][:, :, mid_idx]   # [1, num_val_ids]

# --- Step 1: Collect all targets and predictions ---
all_targets = []
all_preds = []

with torch.no_grad():
    for x_batch, y_batch, target in val_loader:
        x_batch = x_batch.to(config.device)
        pred = model(x_batch, y_teacher=None).cpu()  # [B, 60]
        target = target.cpu()                          # [B, 60]
        
        all_targets.append(target)
        all_preds.append(pred)

all_targets = torch.cat(all_targets)  # [num_val_ids, 60]
all_preds = torch.cat(all_preds)      # [num_val_ids, 60]

# --- Step 2: Denormalize to raw dollar space ---
# val_mid_mean/val_mid_std shape: [1, num_val_ids] -> [num_val_ids, 1] for broadcasting
mid_m = val_mid_mean.squeeze(0).unsqueeze(1).cpu()  # [num_val_ids, 1]
mid_s = val_mid_std.squeeze(0).unsqueeze(1).cpu()   # [num_val_ids, 1]

targets_denorm = all_targets * mid_s + mid_m
preds_denorm = all_preds * mid_s + mid_m

# --- Step 3: Compute R2 and MSE in raw dollar space ---

# per-instrument means
y_mean = targets_denorm.mean(dim=1, keepdim=True)

# sums of squares per instrument
ss_res = ((targets_denorm - preds_denorm) ** 2).sum(dim=1)
ss_tot = ((targets_denorm - y_mean) ** 2).sum(dim=1)

r2_per_inst = 1 - ss_res / ss_tot
final_r2 = r2_per_inst.mean().item()

final_mse = ((preds_denorm - targets_denorm) ** 2).mean().item()

print(f"Denormalized MSE mid: {final_mse:.6f}")
print(f"Denormalized R2 mid:  {final_r2:.6f}")

# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # shift

model_shift.eval()

# Per-instrument normalization stats for VALIDATION set
mid_shift_idx = {col: i for i, col in enumerate(FEATURE_COLS)}["target"]
val_mid_shift_mean = train_scalers_shift["val_feat_means"][:, :, mid_shift_idx]  # [1, num_val_ids]
val_mid_shift_std  = train_scalers_shift["val_feat_stds"][:, :, mid_shift_idx]   # [1, num_val_ids]

# --- Step 1: Collect all targets and predictions ---
all_targets = []
all_preds = []

with torch.no_grad():
    for x_batch, y_batch, target in val_loader_shift:
        x_batch = x_batch.to(config.device)
        pred = model_shift(x_batch, y_teacher=None).cpu()  # [B, 60]
        target = target.cpu()                          # [B, 60]
        all_targets.append(target)
        all_preds.append(pred)

all_targets = torch.cat(all_targets)  # [num_val_ids, 60]
all_preds = torch.cat(all_preds)      # [num_val_ids, 60]

# --- Step 2: Denormalize to raw dollar space ---
# val_mid_mean/val_mid_std shape: [1, num_val_ids] -> [num_val_ids, 1] for broadcasting
mid_shift_m = val_mid_shift_mean.squeeze(0).unsqueeze(1).cpu()  # [num_val_ids, 1]
mid_shift_s = val_mid_shift_std.squeeze(0).unsqueeze(1).cpu()   # [num_val_ids, 1]

targets_denorm = all_targets * mid_shift_s + mid_shift_m
preds_denorm = all_preds * mid_shift_s + mid_shift_m

# --- Step 3: Compute R2 and MSE in raw dollar space ---

# per-instrument means
y_mean = targets_denorm.mean(dim=1, keepdim=True)

# sums of squares per instrument
ss_res = ((targets_denorm - preds_denorm) ** 2).sum(dim=1)
ss_tot = ((targets_denorm - y_mean) ** 2).sum(dim=1)

r2_per_inst = 1 - ss_res / ss_tot
final_r2_shift = r2_per_inst.mean().item()

final_mse_shift = ((preds_denorm - targets_denorm) ** 2).mean().item()

print(f"Denormalized MSE mid: {final_mse_shift:.6f}")
print(f"Denormalized R2 mid:  {final_r2_shift:.6f}")

# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # spread
model_s.eval()

# Per-instrument normalization stats for VALIDATION set
spread_idx = {col: i for i, col in enumerate(FEATURE_COLS)}["target"]
val_spread_mean = train_scalers_s["val_feat_means"][:, :, spread_idx]  # [1, num_val_ids]
val_spread_std  = train_scalers_s["val_feat_stds"][:, :, spread_idx]   # [1, num_val_ids]

# --- Step 1: Collect all targets and predictions ---
all_targets = []
all_preds = []

with torch.no_grad():
    for x_batch, y_batch, target in val_loader:
        x_batch = x_batch.to(config.device)
        pred = model_s(x_batch, y_teacher=None).cpu()  # [B, 60]
        target = target.cpu()                          # [B, 60]
        
        all_targets.append(target)
        all_preds.append(pred)

all_targets = torch.cat(all_targets)  # [num_val_ids, 60]
all_preds = torch.cat(all_preds)      # [num_val_ids, 60]

# --- Step 2: Denormalize to raw dollar space ---
# val_mid_mean/val_mid_std shape: [1, num_val_ids] -> [num_val_ids, 1] for broadcasting
spread_m = val_spread_mean.squeeze(0).unsqueeze(1).cpu()  # [num_val_ids, 1]
spread_s = val_spread_std.squeeze(0).unsqueeze(1).cpu()   # [num_val_ids, 1]

targets_denorm = all_targets * spread_s + spread_m
preds_denorm = all_preds * spread_s + spread_m

# --- Step 3: Compute R2 and MSE in raw dollar space ---

# per-instrument means
y_mean = targets_denorm.mean(dim=1, keepdim=True)

# sums of squares per instrument
ss_res = ((targets_denorm - preds_denorm) ** 2).sum(dim=1)
ss_tot = ((targets_denorm - y_mean) ** 2).sum(dim=1)

r2_per_inst = 1 - ss_res / ss_tot
final_r2_s = r2_per_inst.mean().item()

final_mse_s = ((preds_denorm - targets_denorm) ** 2).mean().item()

print(f"Denormalized MSE spread: {final_mse_s:.6f}")
print(f"Denormalized R2 spread:  {final_r2_s:.6f}")

# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 

metrics_df = pd.DataFrame({
    'data_asset': [data_asset],
    'mse mid': [final_mse],
    'r2 mid': [final_r2],
    'mse mid shift': [final_mse_shift],
    'r2 mid shift': [final_r2_shift],
    'mse spread': [final_mse_s],
    'r2 spread': [final_r2_s],
    'bps_mid': [model_bps.mean().item()],
    'bps_mid_shift': [model_shift_bps.mean().item()],
    'bps_spread': [model_s_bps.mean().item()],
    'bps_mid_spread': [model_ms_bps.mean().item()],
    'bps_twap': [baseline_bps.mean().item()],
    'bps_14_twap': [baseline_14_bps.mean().item()],
    'bps_scheduler': [scheduler_bps.mean().item()]
})

# target_path = base_path / "eval_df.parquet"
# metrics_df.to_parquet(target_path, index=False, engine='pyarrow')
# print(f"Saved eval metrics to: {target_path}")
metrics_df

Denormalized MSE mid: 2041.031860
Denormalized R2 mid:  -18.182364
Denormalized MSE mid: 1856.094238
Denormalized R2 mid:  -15.076655
Denormalized MSE spread: 251.430786
Denormalized R2 spread:  -548.182373


,data_asset,mse mid,r2 mid,mse mid shift,r2 mid shift,mse spread,r2 spread,bps_mid,bps_mid_shift,bps_spread,bps_mid_spread,bps_twap,bps_14_twap,bps_scheduler
0,BTCUSDT,2041.03186,-18.182364,1856.094238,-15.076655,251.430786,-548.182373,1.712027,1.711482,1.739488,1.6219,1.717582,1.323212,1.317433


# Test Predictions

In [ ]:
from utils.test import generate_test_predictions

# --- Execution ---
test_preds, test_id_map = generate_test_predictions(model, config, processor, num_ids=5)

* **The Map:** This `test_id_np` array pairs the model's row index with the actual anonymized ID.

In [ ]:
# Convert test_id_map to numpy (handles both CPU and GPU tensors)
if hasattr(test_id_map, 'cpu'):
    test_id_np = test_id_map.cpu().numpy().astype(np.uint64)
else:
    test_id_np = np.array(test_id_map, dtype=np.uint64)

# Create a quick summary for inspection
print(f"{'Anonymized ID':<25} | {'First Pred (t+1)':<18} | {'Last Pred (t+60)':<18}")
print("-" * 70)

for i in range(len(test_id_np)):  # Look at first 5
    orig_id = test_id_np[i, 1]
    first_p = test_preds[i, 0]
    last_p  = test_preds[i, -1]
    print(f"{str(orig_id):<25} | {first_p:18.6f} | {last_p:18.6f}")

In [ ]:
# 1. Load your data into Polars
# Using test_id_np[:, 1] for our big IDs
id_df = pl.DataFrame({
    "anonymized_id": test_id_np[:, 1],
    "preds": test_preds.tolist()  # This creates a column of lists, each 60 items long
})

# 2. Explode and add the time duration
final_df = (
    id_df.explode("preds")  # This turns each list of 60 into 60 separate rows
    .with_columns(
        # This counts 0 to 59 for every ID
        pl.col("anonymized_id").cum_count().over("anonymized_id").sub(1).alias("step")
    )
    .with_columns(
        # Create the duration starting at 59 minutes
        pl.duration(minutes=59, seconds=pl.col("step")).alias("time_in_hour")
    )
    .select([
        "anonymized_id",
        "time_in_hour",
        pl.col("preds").alias("mid_price_pred")
    ])
)

display(final_df.head(65))

# Final Comparison of Position Prediction

In [ ]:
test_preds.shape

In [ ]:
bla = pd.read_parquet(X_test_path).sort_values(["anonymized_id", "time_in_hour"])
bla["anonymized_id"].unique().shape

In [ ]:
final_df.shape[0] / 60

In [ ]:
final_df["position"].isclose(test_preds[""])